In [20]:
###### Read in the csv files downloaded from dbhydro and summarize annual average ###
import pandas as pd
import os
import pandas as pd
import numpy as np

In [17]:
# File paths
file1 = "/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Salinity/Eastern/MBTS_salinity.csv"
file2 = "/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Salinity/Eastern/MDTS_salinity.csv"

# --- First file ---
df1 = pd.read_csv(file1, skiprows=34)
df1["TIMESTAMP"] = pd.to_datetime(df1["TIMESTAMP"], format="%Y-%m-%d", errors="coerce")
df1 = df1[["TIMESTAMP", "VALUE"]].dropna()

# Convert VALUE column to numeric
df1["VALUE"] = pd.to_numeric(df1["VALUE"], errors="coerce")

df1["Year"] = df1["TIMESTAMP"].dt.year
annual1 = df1.groupby("Year")["VALUE"].mean().reset_index()

output1 = file1.replace(".csv", "_avg.csv")
annual1.to_csv(output1, index=False)

# --- Second file ---
df2 = pd.read_csv(file2, skiprows=34)
df2["TIMESTAMP"] = pd.to_datetime(df2["TIMESTAMP"], format="%Y-%m-%d", errors="coerce")
df2 = df2[["TIMESTAMP", "VALUE"]].dropna()

# Convert VALUE column to numeric
df2["VALUE"] = pd.to_numeric(df2["VALUE"], errors="coerce")

df2["Year"] = df2["TIMESTAMP"].dt.year
annual2 = df2.groupby("Year")["VALUE"].mean().reset_index()

output2 = file2.replace(".csv", "_avg.csv")
annual2.to_csv(output2, index=False)

# Return DataFrames for checking
print("MBTS Annual Means:")
print(annual1.head())
print("\nMDTS Annual Means:")
print(annual2.head())


MBTS Annual Means:
   Year      VALUE
0  2016  24.885035
1  2017  28.345287
2  2018  26.001413
3  2019  35.558230
4  2020  29.813079

MDTS Annual Means:
   Year      VALUE
0  2016        NaN
1  2017  28.932990
2  2018  28.885397
3  2019  35.762340
4  2020  29.091883


In [19]:
import pandas as pd

# File paths
file1 = "/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Salinity/Eastern/MBTS_salinity.csv"
file2 = "/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Salinity/Eastern/MDTS_salinity.csv"

# --- First file ---
df1 = pd.read_csv(file1, skiprows=34)

# Parse TIMESTAMP
df1["TIMESTAMP"] = pd.to_datetime(df1["TIMESTAMP"], errors="coerce")

# Convert VALUE to numeric
df1["VALUE"] = pd.to_numeric(df1["VALUE"], errors="coerce")

# Keep only rows with valid TIMESTAMP and VALUE
df1 = df1[["TIMESTAMP", "VALUE"]].dropna()

# Format TIMESTAMP as mm/dd/yyyy
df1["TIMESTAMP"] = df1["TIMESTAMP"].dt.strftime("%m/%d/%Y")

# Save raw data
output1 = file1.replace(".csv", "_raw.csv")
df1.to_csv(output1, index=False)

# --- Second file ---
df2 = pd.read_csv(file2, skiprows=34)

df2["TIMESTAMP"] = pd.to_datetime(df2["TIMESTAMP"], errors="coerce")
df2["VALUE"] = pd.to_numeric(df2["VALUE"], errors="coerce")
df2 = df2[["TIMESTAMP", "VALUE"]].dropna()
df2["TIMESTAMP"] = df2["TIMESTAMP"].dt.strftime("%m/%d/%Y")

output2 = file2.replace(".csv", "_raw.csv")
df2.to_csv(output2, index=False)

# Return DataFrames for checking
print("MBTS Raw Data:")
print(df1.head())
print("\nMDTS Raw Data:")
print(df2.head())


MBTS Raw Data:
    TIMESTAMP   VALUE
0  08/12/2016  15.431
1  08/13/2016  16.266
2  08/14/2016  13.911
3  08/15/2016  14.727
4  08/16/2016  14.228

MDTS Raw Data:
     TIMESTAMP   VALUE
61  02/01/2017  30.039
90  03/02/2017  31.148
91  03/03/2017  31.115
92  03/04/2017  31.036
93  03/05/2017  31.320


In [21]:
#### Append lat and lon by mapping with the latlon file ######

# --- Step 1: Read annual stats file ---
stats_file = "/Users/bz294/Documents/WhiteZone/Data/Driver_Analysis/Hydro/DBhydro/site/Annual_Hydro_Stats.csv"
stats_df = pd.read_csv(stats_file)

# --- Step 2: Read lat/lon file ---
latlon_file = "/Users/bz294/Documents/WhiteZone/Data/Driver_Analysis/Hydro/DBhydro/DB_wl_latlon.csv"
latlon_df = pd.read_csv(latlon_file)

# --- Step 3: Standardize names for matching ---
stats_df['source_file_upper'] = stats_df['source_file'].str.upper()
latlon_df['Name_upper'] = latlon_df['Name'].str.upper()

# --- Step 4: Function to convert ddmmss.sss to decimal degrees ---
def ddmmss_to_deg(ddmmss):
    """
    Convert ddmmss.sss to decimal degrees
    """
    ddmmss = str(ddmmss).zfill(9)  # Ensure at least 9 digits
    d = int(ddmmss[0:2])
    m = int(ddmmss[2:4])
    s = float(ddmmss[4:])
    return d + m/60 + s/3600

# Convert lat/lon columns
latlon_df['LAT_DEG'] = latlon_df['Lat'].apply(ddmmss_to_deg)
latlon_df['LON_DEG'] = latlon_df['Lon'].apply(ddmmss_to_deg)

# --- Step 5: Merge lat/lon with stats ---
merged_df = stats_df.merge(latlon_df[['Name_upper', 'LAT_DEG', 'LON_DEG']],
                           left_on='source_file_upper',
                           right_on='Name_upper',
                           how='left')

# Drop helper columns
merged_df = merged_df.drop(columns=['source_file_upper', 'Name_upper'])

# --- Step 6: Save updated file ---
output_file = stats_file.replace(".csv", "_with_latlon.csv")
merged_df.to_csv(output_file, index=False)

print(f"Updated annual stats with lat/lon saved to {output_file}")
print(merged_df.head())


Updated annual stats with lat/lon saved to /Users/bz294/Documents/WhiteZone/Data/Driver_Analysis/Hydro/DBhydro/site/Annual_Hydro_Stats_with_latlon.csv
   Year  mean_value  sd_value source_file   LAT_DEG  LON_DEG
0  1992    1.196841  1.299755        MBTS  2.967194  8.17725
1  1993    1.545452  0.347785        MBTS  2.967194  8.17725
2  1994    1.505833  0.405962        MBTS  2.967194  8.17725
3  1995    1.566814  0.372321        MBTS  2.967194  8.17725
4  1996    1.370980  0.379047        MBTS  2.967194  8.17725
